In [1]:
1. Imports + Setup

SyntaxError: invalid syntax (2032116110.py, line 1)

In [2]:
# ==============================
# CSL7110 Assignment 2
# Part 1: k-Grams & Jaccard Similarity
# ==============================

import os
import itertools

In [ ]:
2. Load Documents from data/ Folder

In [3]:
# ------------------------------
# Load Documents
# ------------------------------

def load_document(path):
    with open(path, 'r', encoding='utf-8') as f:
        return f.read().strip()

base_path = "data"

documents = {
    "D1": load_document(os.path.join(base_path, "D1.txt")),
    "D2": load_document(os.path.join(base_path, "D2.txt")),
    "D3": load_document(os.path.join(base_path, "D3.txt")),
    "D4": load_document(os.path.join(base_path, "D4.txt")),
}

print("Documents loaded successfully.")
for name, doc in documents.items():
    print(f"{name} length: {len(doc)} characters")

Documents loaded successfully.
D1 length: 1749 characters
D2 length: 1747 characters
D3 length: 2132 characters
D4 length: 1435 characters


In [ ]:
3. k-Gram Generators

In [4]:
# ------------------------------
# k-Gram Generators
# ------------------------------

def char_kgrams(text, k):
    """
    Generate character k-grams (space counts as character).
    Returns a set (duplicates removed).
    """
    return {text[i:i+k] for i in range(len(text) - k + 1)}

def word_kgrams(text, k):
    """
    Generate word k-grams.
    Returns a set of tuples.
    """
    words = text.split()
    return {tuple(words[i:i+k]) for i in range(len(words) - k + 1)}

In [ ]:
4. Construct Required k-Grams

In [5]:
# ------------------------------
# Construct Required k-Grams
# ------------------------------

char_2grams = {name: char_kgrams(doc, 2) for name, doc in documents.items()}
char_3grams = {name: char_kgrams(doc, 3) for name, doc in documents.items()}
word_2grams = {name: word_kgrams(doc, 2) for name, doc in documents.items()}

print("k-grams constructed successfully.")

for name in documents.keys():
    print(f"{name} -> "
          f"Char2: {len(char_2grams[name])}, "
          f"Char3: {len(char_3grams[name])}, "
          f"Word2: {len(word_2grams[name])}")

k-grams constructed successfully.
D1 -> Char2: 263, Char3: 765, Word2: 279
D2 -> Char2: 262, Char3: 762, Word2: 278
D3 -> Char2: 269, Char3: 828, Word2: 337
D4 -> Char2: 255, Char3: 698, Word2: 232


In [ ]:
5. Jaccard Similarity Function

In [6]:
# ------------------------------
# Jaccard Similarity
# ------------------------------

def jaccard_similarity(set1, set2):
    return len(set1 & set2) / len(set1 | set2)

In [ ]:
6. Compute All Pairwise Similarities

In [7]:
# ------------------------------
# Pairwise Jaccard Similarities
# ------------------------------

pairs = list(itertools.combinations(documents.keys(), 2))

print("========== Character 2-grams ==========")
for d1, d2 in pairs:
    sim = jaccard_similarity(char_2grams[d1], char_2grams[d2])
    print(f"{d1}-{d2}: {sim:.4f}")

print("\n========== Character 3-grams ==========")
for d1, d2 in pairs:
    sim = jaccard_similarity(char_3grams[d1], char_3grams[d2])
    print(f"{d1}-{d2}: {sim:.4f}")

print("\n========== Word 2-grams ==========")
for d1, d2 in pairs:
    sim = jaccard_similarity(word_2grams[d1], word_2grams[d2])
    print(f"{d1}-{d2}: {sim:.4f}")

========== Character 2-grams ==========
D1-D2: 0.9811
D1-D3: 0.8157
D1-D4: 0.6444
D2-D3: 0.8000
D2-D4: 0.6413
D3-D4: 0.6530

========== Character 3-grams ==========
D1-D2: 0.9780
D1-D3: 0.5804
D1-D4: 0.3051
D2-D3: 0.5680
D2-D4: 0.3059
D3-D4: 0.3121

========== Word 2-grams ==========
D1-D2: 0.9408
D1-D3: 0.1823
D1-D4: 0.0302
D2-D3: 0.1737
D2-D4: 0.0303
D3-D4: 0.0161


In [ ]:
1. Prepare 3-gram Sets for D1 & D2

In [8]:
# ------------------------------
# Part 2: MinHash Setup
# ------------------------------

# Use only character 3-grams
set_D1 = char_3grams["D1"]
set_D2 = char_3grams["D2"]

# Universal set of 3-grams
all_3grams = list(set_D1 | set_D2)

print("Total unique 3-grams:", len(all_3grams))

Total unique 3-grams: 772


In [ ]:
2. Create Hash Family

We use standard universal hash:
h(x)=(a∗x+b)modm

In [9]:
import random

def generate_hash_functions(t, m):
    """
    Generate t hash functions of form:
    h(x) = (a*x + b) % m
    """
    hash_functions = []
    for _ in range(t):
        a = random.randint(1, m-1)
        b = random.randint(0, m-1)
        hash_functions.append((a, b))
    return hash_functions

In [ ]:
3. Convert 3-grams to Integers

We must hash strings → integers first.

In [10]:
def gram_to_int(gram):
    return abs(hash(gram))

In [ ]:
4. Compute MinHash Signature

In [11]:
def compute_minhash_signature(gram_set, hash_functions, m):
    signature = []
    
    for (a, b) in hash_functions:
        min_hash = float('inf')
        
        for gram in gram_set:
            x = gram_to_int(gram)
            hash_val = (a * x + b) % m
            if hash_val < min_hash:
                min_hash = hash_val
                
        signature.append(min_hash)
        
    return signature

In [ ]:
5. Estimate Jaccard From Signatures

In [12]:
def estimate_jaccard(sig1, sig2):
    matches = sum(1 for i in range(len(sig1)) if sig1[i] == sig2[i])
    return matches / len(sig1)

In [ ]:
6. Run Experiments for Required t Values

In [13]:
# ------------------------------
# MinHash Experiments
# ------------------------------

m = 20000  # must be > 10000

t_values = [20, 60, 150, 300, 600]

results = []

for t in t_values:
    random.seed(42)  # for reproducibility
    
    hash_funcs = generate_hash_functions(t, m)
    
    sig_D1 = compute_minhash_signature(set_D1, hash_funcs, m)
    sig_D2 = compute_minhash_signature(set_D2, hash_funcs, m)
    
    approx_sim = estimate_jaccard(sig_D1, sig_D2)
    
    results.append((t, approx_sim))
    
    print(f"t = {t} → Approx Jaccard = {approx_sim:.4f}")

t = 20 → Approx Jaccard = 0.9500
t = 60 → Approx Jaccard = 0.9833
t = 150 → Approx Jaccard = 0.9933
t = 300 → Approx Jaccard = 0.9933
t = 600 → Approx Jaccard = 0.9900


In [ ]:
7. Compute exact similarity

In [14]:
exact_sim = jaccard_similarity(set_D1, set_D2)
print("Exact Jaccard (3-grams D1-D2):", round(exact_sim, 4))

Exact Jaccard (3-grams D1-D2): 0.978


In [ ]:
Part3: LSH

In [ ]:
Generate Signatures for ALL Documents (t = 160)

In [15]:
# ------------------------------
# Part 3: LSH Setup
# ------------------------------

t_lsh = 160
m = 20000
random.seed(42)

hash_funcs_lsh = generate_hash_functions(t_lsh, m)

# Compute signatures for all documents (3-grams)
signatures = {}

for name in documents.keys():
    sig = compute_minhash_signature(char_3grams[name], hash_funcs_lsh, m)
    signatures[name] = sig

print("MinHash signatures (t=160) generated for all documents.")

MinHash signatures (t=160) generated for all documents.


In [ ]:
Choose r and b

We must satisfy:
t=r×b=160

Since threshold τ = 0.7
We want S-curve to sharply rise near 0.7.
r = 8
b = 20
Because:
8 × 20 = 160

In [16]:
# Choose band parameters
r = 8
b = 20

print("r =", r, ", b =", b)

r = 8 , b = 20


In [ ]:
Implement Banding

In [17]:
# ------------------------------
# LSH Banding
# ------------------------------

from collections import defaultdict

def lsh(signatures, r, b):
    """
    Perform LSH banding.
    Returns candidate pairs.
    """
    candidates = set()
    doc_names = list(signatures.keys())
    
    for band in range(b):
        buckets = defaultdict(list)
        
        for doc in doc_names:
            start = band * r
            end = start + r
            band_slice = tuple(signatures[doc][start:end])
            buckets[band_slice].append(doc)
        
        # Add pairs from same bucket
        for bucket_docs in buckets.values():
            if len(bucket_docs) > 1:
                for pair in itertools.combinations(bucket_docs, 2):
                    candidates.add(tuple(sorted(pair)))
    
    return candidates

In [ ]:
Get Candidate Pairs

In [18]:
candidate_pairs = lsh(signatures, r, b)

print("Candidate pairs found by LSH:")
for pair in candidate_pairs:
    print(pair)

Candidate pairs found by LSH:
('D1', 'D2')
('D1', 'D3')
('D2', 'D3')


In [ ]:
Compute LSH Probability Formula

In [19]:
# ------------------------------
# S-curve Probability Function
# ------------------------------

def lsh_probability(s, r, b):
    return 1 - (1 - s**r)**b

In [ ]:
Compute Probability for All 6 Pairs

In [20]:
print("========== LSH Probability for Each Pair ==========")

for d1, d2 in pairs:
    exact_sim = jaccard_similarity(char_3grams[d1], char_3grams[d2])
    prob = lsh_probability(exact_sim, r, b)
    print(f"{d1}-{d2}: Similarity = {exact_sim:.4f}, Probability = {prob:.4f}")

========== LSH Probability for Each Pair ==========
D1-D2: Similarity = 0.9780, Probability = 1.0000
D1-D3: Similarity = 0.5804, Probability = 0.2282
D1-D4: Similarity = 0.3051, Probability = 0.0015
D2-D3: Similarity = 0.5680, Probability = 0.1959
D2-D4: Similarity = 0.3059, Probability = 0.0015
D3-D4: Similarity = 0.3121, Probability = 0.0018


In [ ]:
Part 4. Min-Hashing on MovieLens dataset

In [ ]:
1. Load MovieLens Data

In [21]:
# ==============================
# Part 4: MovieLens - Exact Jaccard
# ==============================

import os
from collections import defaultdict

ml_path = os.path.join("movielens", "ml-100k", "u.data")

user_movies = defaultdict(set)

with open(ml_path, 'r') as f:
    for line in f:
        user_id, movie_id, rating, timestamp = line.strip().split('\t')
        user_movies[int(user_id)].add(int(movie_id))

print("Total users:", len(user_movies))

Total users: 943


In [ ]:
2. Exact Jaccard Function (Reuse)
We already defined Jaccard earlier.
Now compute all user pairs ≥ 0.5.

In [22]:
import itertools
import time

start_time = time.time()

exact_pairs = []
users = list(user_movies.keys())

for u1, u2 in itertools.combinations(users, 2):
    set1 = user_movies[u1]
    set2 = user_movies[u2]
    
    intersection = len(set1 & set2)
    union = len(set1 | set2)
    
    sim = intersection / union
    
    if sim >= 0.5:
        exact_pairs.append((u1, u2, sim))

end_time = time.time()

print("Pairs with similarity >= 0.5:", len(exact_pairs))
print("Time taken:", round(end_time - start_time, 2), "seconds")

Pairs with similarity >= 0.5: 10
Time taken: 5.33 seconds


In [ ]:
3. Prepare Exact Pair Set

In [23]:
# Convert exact pairs to set for fast comparison
exact_pair_set = set((u1, u2) for (u1, u2, _) in exact_pairs)

print("Exact pair set size:", len(exact_pair_set))

Exact pair set size: 10


In [ ]:
4. Prepare Movie Universe

In [24]:
# ------------------------------
# Prepare Movie Universe
# ------------------------------

all_movies = set()
for movies in user_movies.values():
    all_movies.update(movies)

all_movies = list(all_movies)
movie_index = {movie: idx for idx, movie in enumerate(all_movies)}

num_movies = len(all_movies)

print("Total unique movies:", num_movies)

Total unique movies: 1682


In [ ]:
5. Efficient MinHash Signature Function

In [25]:
# ------------------------------
# MinHash Signature for Users
# ------------------------------

def generate_hash_functions(t, m):
    hash_functions = []
    for _ in range(t):
        a = random.randint(1, m-1)
        b = random.randint(0, m-1)
        hash_functions.append((a, b))
    return hash_functions


def compute_user_signatures(user_movies, hash_functions, m):
    signatures = {}
    
    for user, movies in user_movies.items():
        sig = []
        movie_ids = [movie_index[m] for m in movies]
        
        for (a, b) in hash_functions:
            min_hash = float('inf')
            for mid in movie_ids:
                hash_val = (a * mid + b) % m
                if hash_val < min_hash:
                    min_hash = hash_val
            sig.append(min_hash)
        
        signatures[user] = sig
    
    return signatures

In [ ]:
6. Evaluate MinHash (Core Experiment Block)
This block will:

Run 5 times

For 50, 100, 200 hashes

Compute false positives & negatives

Report averages

In [26]:
# ------------------------------
# MinHash Evaluation
# ------------------------------

def evaluate_minhash(t, runs=5):
    m = 50000  # large hash space
    
    false_positives = []
    false_negatives = []
    
    for run in range(runs):
        random.seed(run)
        
        hash_funcs = generate_hash_functions(t, m)
        signatures = compute_user_signatures(user_movies, hash_funcs, m)
        
        approx_pairs = set()
        
        users = list(user_movies.keys())
        
        for u1, u2 in itertools.combinations(users, 2):
            sig1 = signatures[u1]
            sig2 = signatures[u2]
            
            matches = sum(1 for i in range(t) if sig1[i] == sig2[i])
            est_sim = matches / t
            
            if est_sim >= 0.5:
                approx_pairs.add((u1, u2))
        
        fp = len(approx_pairs - exact_pair_set)
        fn = len(exact_pair_set - approx_pairs)
        
        false_positives.append(fp)
        false_negatives.append(fn)
        
        print(f"Run {run+1} (t={t}) → FP: {fp}, FN: {fn}")
    
    avg_fp = sum(false_positives) / runs
    avg_fn = sum(false_negatives) / runs
    
    print(f"\nAverage over {runs} runs (t={t}):")
    print("Average False Positives:", avg_fp)
    print("Average False Negatives:", avg_fn)
    
    return avg_fp, avg_fn

In [ ]:
For 50 Hash Functions

In [27]:
evaluate_minhash(50)

Run 1 (t=50) → FP: 131, FN: 1
Run 2 (t=50) → FP: 116, FN: 1
Run 3 (t=50) → FP: 122, FN: 4
Run 4 (t=50) → FP: 115, FN: 4
Run 5 (t=50) → FP: 90, FN: 3

Average over 5 runs (t=50):
Average False Positives: 114.8
Average False Negatives: 2.6


(114.8, 2.6)

In [ ]:
For 100 Hash Functions

In [28]:
evaluate_minhash(100)

Run 1 (t=100) → FP: 88, FN: 1
Run 2 (t=100) → FP: 20, FN: 2
Run 3 (t=100) → FP: 55, FN: 4
Run 4 (t=100) → FP: 42, FN: 2
Run 5 (t=100) → FP: 25, FN: 1

Average over 5 runs (t=100):
Average False Positives: 46.0
Average False Negatives: 2.0


(46.0, 2.0)

In [ ]:
For 200 Hash Functions

In [29]:
evaluate_minhash(200)

Run 1 (t=200) → FP: 9, FN: 1
Run 2 (t=200) → FP: 9, FN: 1
Run 3 (t=200) → FP: 17, FN: 4
Run 4 (t=200) → FP: 12, FN: 3
Run 5 (t=200) → FP: 16, FN: 1

Average over 5 runs (t=200):
Average False Positives: 12.6
Average False Negatives: 2.0


(12.6, 2.0)